# Neuro4ML Coursework

Below, each question that needs a response will be marked by a ⚠️.

In this coursework, you will use a data set of spike trains recorded from monkey motor cortex while it was doing a task involving moving a pointer on a screen. The aim of this coursework is to decode the recorded velocity of the pointer from the neural data using a network of leaky integrate-and-fire neurons that take the recorded spikes as input and give sequences of velocities as outputs. You will train these networks using surrogate gradient descent. If you haven't already looked at it, a great starting point is Friedemann Zenke's [SPyTorch tutorial notebook 1](https://github.com/fzenke/spytorch/blob/main/notebooks/SpyTorchTutorial1.ipynb) (and the rest are worth looking at too).

In this coursework, we are following the general approach of the article ["Machine learning for neural decoding" (Glaser et al. 2020)](https://doi.org/10.1523/ENEURO.0506-19.2020), but using a spiking neural network decoder instead of the statistical and artificial neural network models used in that paper. You can also have a look at the [GitHub repository for the paper](https://github.com/KordingLab/Neural_Decoding). In case you're interested, the data were originally recorded for the paper ["Population coding of conditional probability distributions in dorsal premotor cortex" (Glaser et al. 2018)](https://doi.org/10.1038/s41467-018-04062-6), but you do not need to read this paper to understand this coursework.

The general setup is illustrated in this figure:

![Cartoon of decoder setup](cartoon.png)

You are given an array of ``num_neurons`` spike trains in a variable ``spike_trains``. This variable is a Python list of numpy arrays, each numpy array has a different length and is the recorded times (in seconds) that the corresponding neuron fired a spike. You also have two additional arrays ``vel`` and ``vel_times`` where ``vel`` has shape ``(num_time_points, 2)`` and ``vel_times`` has has shape ``(num_time_points)``. The second axis of ``vel`` has length 2 corresponding to the x and y-components of the recorded velocity.

## Setting up

This section has some basics to get you started.

Let's start by importing some libraries you can make use of. You can solve all the task only using the imports below, but you are welcome to add your own.

In [1]:
import pickle

import numpy as np
from scipy import io
from scipy.signal import savgol_filter

import matplotlib.pyplot as plt

import torch
from torch import nn
import torch.nn.functional as tnnf
mse = nn.MSELoss()

from tqdm.notebook import tqdm

ms = 1e-3 # use this constant so you can write e.g. 1*ms for a time

In [2]:
# Reproducible experiments and notebook-native inline figures.
torch.manual_seed(42)
np.random.seed(42)
import matplotlib
matplotlib.use('Agg')
# Capture every generated plot as a PNG (reliable in headless environments).
from pathlib import Path
_plot_number = [0]
def save_plot(*args, **kwargs):
    _plot_number[0] += 1
    plt.savefig(Path('plots') / f'plot_{_plot_number[0]:02d}.png', dpi=115, bbox_inches='tight')
    plt.close()
Path('plots').mkdir(exist_ok=True)
plt.show = save_plot

You already have a copy of the raw data, but for your information, here is where the original can be downloaded.

In [3]:
import urllib.request, zipfile, os
filename = 's1_data_raw.mat'
if not os.path.exists(filename):
    urllib.request.urlretrieve('https://www.dropbox.com/sh/n4924ipcfjqc0t6/AACPWjxDKPEzQiXKUUFriFkJa?dl=1', 'data.zip')
    with zipfile.ZipFile('data.zip') as z:
        z.extract(filename)

## Task 1: Load and plot the data

The code below first loads the raw data, which is stored as a Matlab file, and then extracts the three arrays ``spike_times``, ``vel`` and ``vel_times``.

In [4]:
# Load the raw data
data = io.loadmat('s1_data_raw.mat') # a matlab file!
spike_times = [st[:, 0] for st in data['spike_times'].ravel()] # a list of arrays of spike times in seconds, one for each neuron, spike times in seconds
vel = data['vels'] # velocity data shape (num_time_points, 2) for (x, y) coordinates
vel_times = data['vel_times'].squeeze() # times the velocities were recorded

### Task 1A: Preprocess and compute basic statistics

In this task, you will preprocess the data, extract some basic statistics from it.

1. ⚠️ Whiten the recorded velocities (i.e. transform them so that their mean is 0 and standard deviation is 1).
2. ⚠️ Compute and print out the number of neurons and number of spikes recorded.
3. ⚠️ Compute and print out the duration of the experiment in seconds and/or minutes.
4. ⚠️ Compute and print out the sampling rate at which spikes were recorded (or find the information in the corresponding paper).
5. ⚠️ Compute and print out the sampling rate at which velocities were recorded (or find the information in the corresponding paper).

Note that the spikes and velocities were recorded with different equipment and so they have different sampling rates. Think about how you can estimate these sampling rates from the recorded data (or look it up in the paper).

In [5]:

# 1. Whiten the recorded velocities
vel_raw = vel.copy()
vel_mean, vel_std = vel_raw.mean(axis=0), vel_raw.std(axis=0)
vel = (vel_raw - vel_mean) / vel_std  # normalise each velocity component independently.
print('Normalised velocity mean:', vel.mean(axis=0).round(5))
print('Normalised velocity std:', vel.std(axis=0).round(5))

# 2. Compute and print the number of neurons and spikes recorded
num_neurons = len(spike_times)
num_spikes = sum(len(st) for st in spike_times)
print(f'Neurons: {num_neurons}; spikes: {num_spikes:,}')

# 3. Compute and print the duration of the experiment
duration = float(max(max(st[-1] for st in spike_times), vel_times[-1]) -
                 min(min(st[0] for st in spike_times), vel_times[0])) # use the latest and earliest timestamps across spikes and velocities
print(f'Duration: {duration:.2f} s ({duration/60:.2f} min)')

# 4. Compute and print the estimated spike sampling rate.
spike_intervals = np.concatenate([
    np.diff(st) for st in spike_times if len(st) > 1 # calculate intervals between consecutive spikes for each neuron 
])
min_interval = np.min(spike_intervals[spike_intervals > 0]) # smallest interval between spikes
spike_fs = round(1.0 / min_interval) # frequency is 1 / interval 
print(f'Estimated spike timestamp rate: {spike_fs} Hz')

# 5. Compute and print the velocity sampling rate.
velocity_fs = 1.0 / np.median(np.diff(vel_times)) # velocity samples are regularly spaced so use median time interval
print(f'Velocity sampling rate: {velocity_fs:.2f} Hz')

Normalised velocity mean: [-0.  0.]
Normalised velocity std: [1. 1.]
Neurons: 52; spikes: 1,480,032
Duration: 3071.32 s (51.19 min)
Estimated spike timestamp rate: 612 Hz
Velocity sampling rate: 100.00 Hz


### Task 1B: Plot the data

In this task, you will plot the data to get a feeling for what it is like (an important step in any modelling).

1. ⚠️ Plot the spike times as a raster plot (black dots at x-coordinates the time of the spike, and y-coordinates the index of the neuron). Plot this both for the whole data set and for the period from 1000 to 1010 seconds.
2. ⚠️ Plot the x- and y-coordinates of the velocities. Plot this both for the whole data set and for the same period as above for the spikes.
3. ⚠️ Compute the mean firing rate (number of spikes per second) for each neuron and display as a bar chart.
4. ⚠️ Plot the velocities as a curve in (x, y) space, emphasising the part of the velocity curve for the period above.

You can use the template below to get you started.

In [6]:
# plt.figure(figsize=(12, 5))

# # Plot all spikes
# ax = plt.subplot(231)
# # ...
# plt.ylabel('Neuron index')
# # Plot all x- and y- components of the velocities
# plt.subplot(234, sharex=ax)
# # ...
# plt.xlabel('Time (s)')
# plt.ylabel('velocity')
# plt.legend(loc='best')
# # Plot spikes at times t=1000 to t=1010
# ax = plt.subplot(232)
# # ...
# # Plot velocities at times t=1000 to t=1010
# plt.subplot(235, sharex=ax)
# plt.xlabel('Time (s)')

# # Compute firing rates for each neuron and plot as a histogram
# plt.subplot(233)
# firing_rates = ...
# plt.barh(range(len(spike_times)), firing_rates, height=1)
# plt.xlabel('Firing rate (sp/s)')
# plt.ylabel('Neuron index')

# # Plot all velocities as points in (x, y)-plane as a continuous curve
# # Emphasise the region from t=1000 to t=1010 with a different colour
# plt.subplot(236)
# # ...
# plt.xlabel('X velocity')
# plt.ylabel('Y velocity')

# plt.tight_layout();

In [7]:
# Task 1B: full-record and zoomed neural/behavioural visualisation.
fig, ax = plt.subplots(2, 3, figsize=(17, 9))
for i, st in enumerate(spike_times):
    ax[0, 0].plot(st, np.full(len(st), i), ',k', alpha=.15)
    near = st[(st >= 1000) & (st <= 1010)]
    ax[0, 1].plot(near, np.full(len(near), i), '.k', ms=1.5)
ax[0, 0].set(title='All recorded spikes', xlabel='Time (s)', ylabel='Neuron')
ax[0, 1].set(title='Spikes: 1000–1010 s', xlabel='Time (s)', ylabel='Neuron', xlim=(1000, 1010))
ax[1, 0].plot(vel_times, vel[:, 0], lw=.5, label='x')
ax[1, 0].plot(vel_times, vel[:, 1], lw=.5, label='y')
ax[1, 0].set(xlabel='Time (s)', ylabel='Normalised velocity', title='Recorded velocity')
ax[1, 0].legend()
mask = (vel_times >= 1000) & (vel_times <= 1010)
ax[1, 1].plot(vel_times[mask], vel[mask, 0], label='x')
ax[1, 1].plot(vel_times[mask], vel[mask, 1], label='y')
ax[1, 1].set(xlabel='Time (s)', ylabel='Normalised velocity', title='Velocity: 1000–1010 s')
ax[1, 1].legend()
firing_rates = np.array([len(st) / duration for st in spike_times])
ax[0, 2].barh(np.arange(num_neurons), firing_rates, height=.85)
ax[0, 2].set(xlabel='Spikes/s', ylabel='Neuron', title='Mean firing rate by neuron')
ax[1, 2].plot(vel[:, 0], vel[:, 1], lw=.25, alpha=.35, label='Entire recording')
ax[1, 2].plot(vel[mask, 0], vel[mask, 1], lw=1.5, label='1000–1010 s')
ax[1, 2].set(xlabel='X velocity', ylabel='Y velocity', title='Velocity-space trajectory')
ax[1, 2].legend()
plt.tight_layout(); plt.show()
print(f'Firing-rate mean={firing_rates.mean():.2f} Hz, median={np.median(firing_rates):.2f} Hz')

Firing-rate mean=9.27 Hz, median=6.31 Hz


## Task 2: Divide data into test/train and batches

In this section, you will divide the data up into non-overlapping test, train and (optionally) validation sets.

### Task 2A: Why not just use continuous time ranges?

In the Glaser et al. paper linked to above, they simply divide the range so that the first 70% of the data is used for training, the next 15% for validation and the last 15% for testing. This is a standard thing to do.

1. ⚠️ Compute the speeds from the velocity components, plot them and compute some statistics to show why this is a bad idea for this dataset. To make the plot clear, you might want to take averages over some window rather than plotting every point.

In [8]:
# Task 2A: quantify distribution shift if time is split consecutively.
speed = np.linalg.norm(vel, axis=1)
cut1, cut2 = int(.70 * len(speed)), int(.85 * len(speed))
continuous_splits = {'first 70%': speed[:cut1], 'next 15%': speed[cut1:cut2],
                     'last 15%': speed[cut2:]}
print('Sequential time split:')
for name, s in continuous_splits.items():
    print(f'  {name:>10}: mean speed={s.mean():.3f}, std={s.std():.3f}, '
          f'90th percentile={np.percentile(s,90):.3f}, fraction speed<0.5={(s < .5).mean():.3f}')
rolling_window = int(round(5 * velocity_fs))
smooth_speed = np.convolve(speed, np.ones(rolling_window)/rolling_window, mode='same')
plt.figure(figsize=(13, 3))
plt.plot(vel_times, smooth_speed, lw=.7)
for frac in (.7, .85):
    plt.axvline(vel_times[int(frac*len(vel_times))], ls='--', color='k')
plt.xlabel('Time (s)'); plt.ylabel('Speed (normalised units)')
plt.title('Five-second moving mean speed: contiguous splits are not exchangeable')
plt.tight_layout(); plt.show()

Sequential time split:
   first 70%: mean speed=1.040, std=1.017, 90th percentile=2.435, fraction speed<0.5=0.382
    next 15%: mean speed=0.929, std=0.916, 90th percentile=2.183, fraction speed<0.5=0.413
    last 15%: mean speed=0.955, std=0.919, 90th percentile=2.195, fraction speed<0.5=0.401


**Interpretation.** The behaviour can be non-stationary across the recording (for example, periods of activity versus inactivity). A contiguous split can therefore place distinct speed distributions in training, validation and test sets, making generalisation estimates depend on time period. In Task 2B, whole time blocks are allocated across the recording instead. This does not eliminate temporal dependence, so all samples from a block remain in the same split.

### Task 2B: use equally distributed segments for training, test and validation

1. ⚠️ Divide the data up into segments and use some fraction of these segments for training, test and validation, ensuring these segments do not overlap but that they are equally distributed throughout the data.
2. ⚠️ Write a generator function (see below) ``batched_data`` that iterates over your data in randomly ordered segments of a given length, returning it in batches. The function should have arguments that determine the fraction of the data to use, the simulation time step that will be used, the length (in seconds) of each batch, and the batch size (you may add additional arguments if you wish). The function should return a pair of arrays ``(x, y)``. The array ``x`` has shape ``(batch_size, num_neurons, num_time_points)`` containing the spike times as an array where a zero indicates no spike and 1 indicates a spike. Here ``num_time_points`` is the number of time points in the batch measured at the sampling rate of the simulation time step, not the number of time points in the data as a whole, nor at the spike or velocity sampling rate. The array ``y`` has shape ``(batch_size, 2, num_time_points)`` containing the velocities at the same time points as the spikes. You will need to use some sort of interpolation to get the velocities at these times.
3. ⚠️ Plot a sample of spike times and velocities for a random batch of length 1 second and ``batch_size=4``.

#### Note on generator functions

Generator functions are an advanced feature of Python that makes it easy to iterate over complicated datasets. The general syntax is just a standard function that uses the keyword ``yield`` instead of ``return`` to return data, which allows it to return data multiple times. You can iterate over the values returned by a generator function instead of just calling it. Here's an example:

In [9]:
def gimme_some_multiples(n):
    yield n*1
    yield n*2
    yield n*3

for x in gimme_some_multiples(3):
    print(x)

3
6
9


And another:

In [10]:
def gimme_some_more_multiples(n, how_many):
    for i in range(how_many):
        yield n*(i+1)

for x in gimme_some_more_multiples(5, 4):
    print(x)

5
10
15
20


#### Template

You can use the following template but you may want to define some additional helper functions to simplify your code and that you can re-use later.

In [11]:
# # Template

# # Set up training / validation / testing ranges
# # ...

# # Generator function that yields batches
# def batched_data(range_to_use, dt=1e-3, length=1, batch_size=64):
#   pass # ...
#   for batch_idx in range(num_batches):
#     x = torch.zeros((batch_size, num_neurons, num_time_points))
#     y = torch.zeros((batch_size, 2, num_time_points))
#     for b in range(batch_size):
#       pass # ...
#     yield x, y

# # Plot a sample of data

# x, y = next(batched_data(...)) # this just gets the first item of an iterable

# plt.figure(figsize=(12, 5))
# for b in range(4):
#   # Plot spikes for this batch index
#   ax = plt.subplot(2, 4, b+1)
#   # ...
#   plt.ylabel('Neuron index')
#   plt.title(f'Batch index {b}')
#   # Plot velocities for this batch index
#   plt.subplot(2, 4, b+5, sharex=ax)
#   plt.xlabel('Time index')
#   plt.ylabel('velocity')
#   if b==0:
#     plt.legend(loc='best')
# plt.tight_layout();

In [12]:
# Task 2B: disjoint 20-second BLOCKS assigned across the entire experiment.
# A fixed seed makes the split reproducible. Overlapping windows never cross splits.
rng = np.random.default_rng(42)
block_seconds = 20.0
t_min, t_max = float(vel_times[0]), float(vel_times[-1])
block_starts = np.arange(t_min, t_max-block_seconds, block_seconds)
block_ids = rng.permutation(len(block_starts))
n_train = int(.70 * len(block_ids)); n_val = int(.15 * len(block_ids))
training_range = np.sort(block_starts[block_ids[:n_train]])
validation_range = np.sort(block_starts[block_ids[n_train:n_train+n_val]])
testing_range = np.sort(block_starts[block_ids[n_train+n_val:]])
print('Disjoint blocks (train, validation, test):',
      len(training_range), len(validation_range), len(testing_range))
assert not (set(training_range) & set(validation_range) or
            set(training_range) & set(testing_range) or
            set(validation_range) & set(testing_range))

def build_window(start, length=1., dt=.01):
    """Create aligned binary spike raster and interpolated velocity targets.

    Each spike is assigned to floor((spike_time-start)/dt). Multiple spikes in
    one bin still produce one binary event, as required by the assignment.
    """
    T = int(round(length / dt))
    ts = start + np.arange(T)*dt
    xs = np.zeros((num_neurons, T), dtype=np.float32)
    for i, st in enumerate(spike_times):
        lo, hi = np.searchsorted(st, [start, start + T*dt])
        bins = np.floor((st[lo:hi]-start)/dt).astype(np.int64)
        bins = bins[(bins >= 0) & (bins < T)]
        xs[i, bins] = 1.0
    ys = np.vstack([np.interp(ts, vel_times, vel[:, c]) for c in range(2)]).astype(np.float32)
    return xs, ys

def batched_data(range_to_use, dt=.001, length=1., batch_size=64,
                 max_num_batches=None, seed=None):
    """Yield randomized non-overlapping windows from the selected time blocks.

    `range_to_use` contains starts of disjoint block_seconds-sized blocks.
    Samples from different splits never share underlying time intervals.
    """
    local_rng = np.random.default_rng(seed)
    candidate_starts = []
    for block_start in range_to_use:
        # Randomised offset per block, then full non-overlapping windows.
        offset = local_rng.uniform(0, min(length, block_seconds-length))
        candidate_starts.extend(np.arange(block_start+offset,
                                           block_start+block_seconds-length+1e-8, length))
    candidate_starts = np.asarray(candidate_starts)
    local_rng.shuffle(candidate_starts)
    n_batches = len(candidate_starts) // batch_size
    if max_num_batches is not None:
        n_batches = min(n_batches, max_num_batches)
    for batch_idx in range(n_batches):
        starts = candidate_starts[batch_idx*batch_size:(batch_idx+1)*batch_size]
        pairs = [build_window(start, length, dt) for start in starts]
        x = torch.from_numpy(np.stack([p[0] for p in pairs]))
        y = torch.from_numpy(np.stack([p[1] for p in pairs]))
        yield x, y

# Sample 1-second batches at the original 1-ms resolution.
x_example, y_example = next(batched_data(training_range, dt=.001, length=1,
                                         batch_size=4, max_num_batches=1, seed=42))
fig, ax = plt.subplots(2,4,figsize=(14,5),sharex='col')
for b in range(4):
    ids, timesteps = torch.where(x_example[b] > 0)
    ax[0,b].scatter(timesteps,ids,s=.4,color='k')
    ax[0,b].set_title(f'Batch item {b}')
    ax[1,b].plot(y_example[b,0],label='x'); ax[1,b].plot(y_example[b,1],label='y')
    ax[1,b].set_xlabel('Simulation timestep')
    if b==0:
        ax[0,b].set_ylabel('Neuron'); ax[1,b].set_ylabel('Velocity'); ax[1,b].legend()
plt.tight_layout(); plt.show()
print('Batch shapes:',tuple(x_example.shape),tuple(y_example.shape))

Disjoint blocks (train, validation, test): 107 22 24
Batch shapes: (4, 52, 1000) (4, 2, 1000)


## Task 3: Spiking neural network model

In this task, you will write code to simulate a spiking neural network that can be trained using surrogate gradient descent, as in lectures. The neuron model will have some extra features beyond what you have seen in lectures and in the SPyTorch tutorial.

### Surrogate gradient descent spike function

Below is the code for the surrogate gradient descent function from lectures. You can use it as is, although note that there is a hyperparameter (scale) that you can experiment with if you choose.

In [13]:
class SurrogateHeaviside(torch.autograd.Function):
    """
    Here we implement our spiking nonlinearity which also implements
    the surrogate gradient. By subclassing torch.autograd.Function,
    we will be able to use all of PyTorch's autograd functionality.
    Here we use the normalized negative part of a fast sigmoid
    as this was done in Zenke & Ganguli (2018).
    """

    scale = 100.0 # controls steepness of surrogate gradient

    @staticmethod
    def forward(ctx, input):
        """
        In the forward pass we compute a step function of the input Tensor
        and return it. ctx is a context object that we use to stash information which
        we need to later backpropagate our error signals. To achieve this we use the
        ctx.save_for_backward method.
        """
        ctx.save_for_backward(input)
        out = torch.zeros_like(input)
        out[input > 0] = 1.0
        return out

    @staticmethod
    def backward(ctx, grad_output):
        """
        In the backward pass we receive a Tensor we need to compute the
        surrogate gradient of the loss with respect to the input.
        Here we use the normalized negative part of a fast sigmoid
        as this was done in Zenke & Ganguli (2018).
        """
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad = grad_input/(SurrogateHeaviside.scale*torch.abs(input)+1.0)**2
        return grad

# here we overwrite our naive spike function by the "SurrogateHeaviside" nonlinearity which implements a surrogate gradient
surrogate_heaviside  = SurrogateHeaviside.apply

### Task 3A: Single layer simulation code

1. ⚠️ Write modular code to simulate a layer of leaky integrate-and-fire spiking neurons compatible with autodifferentiation with PyTorch. The new feature compared to what you have seen before is that the time constant parameter $\tau$ should be different for each neuron, and be trainable.

You can write your simulator as a function or class. My recommendation is to derive from the ``nn.Module`` class of PyTorch, but you can do it how you like. The code should accept an input batch of spikes ``x`` of shape ``(batch_size, num_input_neurons, num_time_points)`` and values 0 or 1 (as in the ``batched_data`` generator function above). The code should have the option to produce either spiking or non-spiking output. In both cases, the output should be an array ``y`` of shape ``(batch_size, num_output_neurons, num_time_points)``. In the case of spiking output, the values of ``y`` should be 0s and 1s, and in the case of non-spiking output they should be the membrane potential values. You may also want to write an additional class to handle multiple layers of spiking neural networks for subsequent tasks.

Your code should include initialisation of the weight matrices and time constants, and add additional hyperparameters for this initialisation. I used a spread of time constants from 20-100 ms for spiking neurons and from 200 to 1000 ms for non-spiking neurons and it worked OK, but I didn't do an extensive hyperparameter search and you can probably improve on this.

<!-- I would recommend approaching this and the following sections as follows:

1. Write simulation code for a single layer, non-spiking neural network first. This code is simpler and will train fast (under 3 minutes on Colab). Attempt as much of the remaining tasks as possible using only this.
2. Add the ability for spiking and test that your code produces reasonable output but don't try to train it yet.
3. Add the ability to plot spiking hidden layers and try to get a reasonable initialisation of the network.
4. Start training the spiking neural network. Your final run will probably take a long time to train but you should build up to that by seeing how well the training curves improve for fewer epochs, batch sizes, etc. -->

You can use the template below if you want to use the class-based approach.

You may also find it useful to write a class that chains together a group of layers for later parts of the courseowrk.

In [14]:
# class SNNLayer(nn.Module):
#   def __init__(self, n_in, n_out, spiking=True):
#     super(SNNLayer, self).__init__()
#     self.n_in = n_in
#     self.n_out = n_out
#     self.spiking = spiking
#     # Store weights as a trainable parameter
#     self.w = nn.Parameter(torch.ones((n_in, n_out)))
#     # ...

#   def forward(self, x):
#     batch_size, num_neurons, num_time_points = x.shape
#     # ...
#     return y

In [15]:
# Task 3A: LIF cell with per-neuron trainable decay time constants.
class SNNLayer(nn.Module):
    def __init__(self, n_in, n_out, spiking=True, dt=.01,
                 tau_range=None, weight_scale=.1, threshold=1.):
        super().__init__()
        self.n_in, self.n_out, self.spiking, self.dt = n_in,n_out,spiking,dt
        self.threshold = threshold
        if tau_range is None:
            tau_range = (.02,.1) if spiking else (.2,1.)
        self.w = nn.Parameter(torch.randn(n_in,n_out)*weight_scale)
        tau_init = torch.linspace(*tau_range,n_out)
        self.log_tau = nn.Parameter(torch.log(tau_init))
        self.last_membrane = None
        self.last_spikes = None

    @property
    def tau(self):
        return self.log_tau.exp().clamp(min=.001)

    def forward(self,x,record=False):
        B,_,T=x.shape
        # Matrix multiplication performed once per sequence, not once per step.
        current = torch.einsum('bit,io->bot',x,self.w)
        alpha = torch.exp(-self.dt/self.tau).unsqueeze(0)
        u = torch.zeros(B,self.n_out,device=x.device,dtype=x.dtype)
        result,volts = [],[]
        for t in range(T):
            u = alpha*u + current[:,:,t]
            volts.append(u)
            if self.spiking:
                z = surrogate_heaviside(u-self.threshold)
                result.append(z)
                # Subtractive reset (the residual potential is preserved).
                u = u - self.threshold*z
            else:
                result.append(u)
        y = torch.stack(result,dim=-1)
        if record:
            self.last_membrane = torch.stack(volts,dim=-1).detach()
            self.last_spikes = y.detach() if self.spiking else None
        return y

class SNNNetwork(nn.Module):
    def __init__(self, sizes,dt=.01,hidden_spiking=True,weight_scale=.1):
        super().__init__()
        self.layers = nn.ModuleList([
            SNNLayer(sizes[i],sizes[i+1],
                     spiking=(i<len(sizes)-2 and hidden_spiking),
                     dt=dt,weight_scale=weight_scale)
            for i in range(len(sizes)-1)])
        self.fname = 'network'
    def forward(self,x,record=False):
        for layer in self.layers:
            x = layer(x,record=record)
        return x

def parameters_are_finite(model):
    return all(torch.isfinite(p).all() for p in model.parameters())

# Basic dimensions, differentiability and independent taus.
check_layer = SNNLayer(3,4,spiking=True,dt=.001)
check_in = torch.rand(2,3,20)
check_out = check_layer(check_in)
assert check_out.shape == (2,4,20)
assert torch.all((check_out==0) | (check_out==1))
(check_out.mean()).backward()
assert check_layer.w.grad is not None and check_layer.log_tau.grad is not None
print('Spiking layer shape and autograd checks passed.')

Spiking layer shape and autograd checks passed.


### Task 3B: Verify your code

You will verify your code by simulating a network consisting of:

* One (non-simulated) input neuron firing at a constant rate of 50 sp/s for a total duration of 1 second simulated with a time step of ``dt=1*ms``.
* Two simulated, spiking neurons which each receive spikes from the input neuron with a weight of 0.5. The two simulated neurons should have time constants of 20 ms and 100 ms.

You should:

1. ⚠️ Run the simulation, recording the membrane potentials and spike times of the simulated neurons.
2. ⚠️ Plot the membrane potentials and spike times of the simulated neurons. Print the spike counts of the neurons. The first neuron should not fire any spikes.
3. ⚠️ Write down an analytic solution for the first simulated neuron (that doesn't fire any spikes).
4. ⚠️ Plot your analytic solution on the same graph as the simulated one to show that it is approximately equal.

**Analytic solution (Task 3B).** With a zero initial membrane potential, a membrane time constant \(\tau\), input spikes at \(t_j\), impulse weight \(w=0.5\), and no reset, the solution to exponential leak between impulses is

\[
u(t)=0.5\sum_{t_j\leq t}\exp\left(-\frac{t-t_j}{\tau}\right).
\]

The first neuron has \(\tau=20\,\mathrm{ms}\); with regularly spaced spikes every 20 ms its steady-state post-input maximum is \(0.5/(1-e^{-1})\approx0.791<1\), so it never reaches the unit threshold. The 100 ms neuron accumulates spikes and crosses threshold. The simulator uses the exact exponential decay per time step; its no-reset voltage should match this analytic expression to numerical precision.

In [16]:
# Task 3B: validate both cell dynamics and spike counts.
dt_verify = .001
T_verify = 1000
stim = torch.zeros(1,1,T_verify)
stim[0,0,::20] = 1.  # 50 spikes per second
verify_layer = SNNLayer(1,2,spiking=True,dt=dt_verify,
                        tau_range=(.02,.10),weight_scale=0.)
with torch.no_grad():
    verify_layer.w.fill_(.5)
    verify_layer.log_tau.copy_(torch.log(torch.tensor([.02,.10])))
    verify_out = verify_layer(stim,record=True)
counts = verify_out.sum(dim=(0,2)).numpy().astype(int)
assert counts[0] == 0, 'The 20-ms neuron must remain below threshold.'
print('Output spike counts (20 ms, 100 ms):',counts)
t_verify=np.arange(T_verify)*dt_verify
input_spike_times=np.arange(0,1,.02)
analytic = .5 * np.exp(-np.maximum(t_verify[:,None]-input_spike_times[None,:],0)/.02)
analytic *= (t_verify[:,None] >= input_spike_times[None,:])
analytic = analytic.sum(axis=1)
simulated = verify_layer.last_membrane[0,0].numpy()
print('Maximum absolute analytical error:',np.max(np.abs(analytic-simulated)))
assert np.allclose(analytic,simulated,atol=1e-5)
fig,axes=plt.subplots(2,1,figsize=(12,6),sharex=True)
for i,tau in enumerate((.02,.10)):
    axes[i].plot(t_verify,verify_layer.last_membrane[0,i].numpy(),label='Pre-reset simulation')
    if i==0: axes[i].plot(t_verify,analytic,'--',label='Analytic (no spikes)')
    axes[i].scatter(t_verify[verify_out[0,i].numpy()>0],
                    np.ones(counts[i]),marker='|',s=90,label=f'Spikes ({counts[i]})')
    axes[i].axhline(1,color='k',ls=':',alpha=.5)
    axes[i].set(ylabel='Membrane',title=f'Time constant {tau*1000:.0f} ms')
    axes[i].legend()
axes[-1].set_xlabel('Time (s)'); plt.tight_layout(); plt.show()

Output spike counts (20 ms, 100 ms): [ 0 20]
Maximum absolute analytical error: 3.865632594024504e-07


## Task 4: Evaluating fit to data

Now that we have our simulation code, we get back to fitting this model to data.

1. ⚠️ Write code that takes a network and testing set as input and returns the mean loss over the testing data. The loss is the mean squared error of the output of the network compared to the target data. You may also find it helpful to compute the null loss, which is the loss you would get if you just output all zeros. After training, you should be able to do better than this!
2. ⚠️ Write code that plots some of the internal outputs of the network, for example to show you the spikes produced by hidden layers, calculate their firing rates, etc.
3. ⚠️ Initialise a network with one hidden layer of 100 spiking neurons and one output layer of 2 non-spiking neurons. Run this on a random sample of the data of length 1 and plot the input spikes, hidden layer spikes, output x- and y-velocities, and the data x- and y-velocities. For each spiking layer compute the firing rates of each neuron and plot them as a histogram.

In the next task, you can use this to initialise your networks in a reasonable state. Hidden layers should fire spikes at rates that are not too low and not too high. I aimed for an average firing rate in the range 20-100 and it worked well, but you can experiment with other options. The output layer should give values that are roughly in the right range as the data (i.e. it shouldn't go to +/- 100 if the data is going to only +/- 4). If you look at the spike trains of your hidden layer and all the neurons at initialisation are doing exactly the same thing then it's probably not going to learn very well, so try out some different weight initialisations to see if you can do better.

Print the value of the loss (and optionally the null loss) for your untrained network, to give you an idea of the baseline.

You may want to wrap your evaluation code in a ``with`` statement like below to stop PyTorch from computing gradients when evaluating (unnecessary and expensive):

```python
with torch.no_grad():
    ...
    # whatever you do here won't compute any gradients
```

In [17]:
# Task 4: MSE evaluation with all three splits kept separate.
@torch.no_grad()
def evaluate_network(net, range_to_use, length=1.,batch_size=16,dt=.01,
                     max_num_batches=10,seed=111):
    net.eval()
    squared_error,total_sum_sq,n=0.,0.,0
    for x,y in batched_data(range_to_use,dt=dt,length=length,
                            batch_size=batch_size,max_num_batches=max_num_batches,
                            seed=seed):
        prediction=net(x)
        squared_error += tnnf.mse_loss(prediction,y,reduction='sum').item()
        total_sum_sq += (y*y).sum().item()
        n += y.numel()
    if n==0: raise ValueError('No complete batches in the requested split.')
    return squared_error/n,total_sum_sq/n

@torch.no_grad()
def plot_network_activity(net,range_to_use,dt=.01,length=1.,seed=2):
    net.eval()
    x,y = next(batched_data(range_to_use,dt=dt,length=length,
                            batch_size=1,max_num_batches=1,seed=seed))
    out=net(x,record=True)
    fig,axes=plt.subplots(3+sum(l.spiking for l in net.layers),1,
                          figsize=(12,7),sharex=True)
    idx,ts=torch.where(x[0]>0)
    axes[0].scatter(ts*dt,idx,s=.5,color='k');axes[0].set_ylabel('Input neuron')
    j=1
    for layer in net.layers:
        if layer.spiking:
            idx,ts=torch.where(layer.last_spikes[0]>0)
            axes[j].scatter(ts*dt,idx,s=1,color='k')
            fr=layer.last_spikes[0].sum(-1).numpy()/length
            axes[j].set_ylabel('Hidden neuron')
            print('Hidden firing rates (Hz): mean',fr.mean(),'min',fr.min(),'max',fr.max())
            plt.figure(figsize=(6,2));plt.hist(fr,bins=20)
            plt.xlabel('Hidden neuron firing rate (Hz)');plt.ylabel('Count');plt.show()
            j+=1
    axes[j].plot(np.arange(y.shape[-1])*dt,y[0,0],label='x actual',alpha=.8)
    axes[j].plot(np.arange(y.shape[-1])*dt,out[0,0],label='x predicted',alpha=.8)
    axes[j].legend();axes[j].set_ylabel('X velocity');j+=1
    axes[j].plot(np.arange(y.shape[-1])*dt,y[0,1],label='y actual',alpha=.8)
    axes[j].plot(np.arange(y.shape[-1])*dt,out[0,1],label='y predicted',alpha=.8)
    axes[j].legend();axes[j].set_ylabel('Y velocity')
    axes[-1].set_xlabel('Time (s)');plt.tight_layout();plt.show()
    # Input firing-rate histogram (input is also a spiking layer).
    input_rates=x[0].sum(-1).numpy()/length
    plt.figure(figsize=(6,2));plt.hist(input_rates,bins=20)
    plt.xlabel('Input neuron firing rate (Hz)');plt.ylabel('Count');plt.show()

training_dt=.01  # 10-ms simulation step reduces training cost; 1 ms used in Task 3B.
initial_spiking=SNNNetwork([num_neurons,100,2],dt=training_dt,weight_scale=.4)
with torch.no_grad():
    initial_spiking.layers[-1].w.mul_(.025)
plot_network_activity(initial_spiking,training_range,dt=training_dt)
base_loss,base_null=evaluate_network(initial_spiking,validation_range,dt=training_dt)
print(f'Untrained 100-spiking-hidden net: MSE {base_loss:.4f}; null {base_null:.4f}')

Hidden firing rates (Hz): mean 11.78 min 0.0 max 47.0
Untrained 100-spiking-hidden net: MSE 1.2487; null 1.0231


## Task 5: Training

At last, we get to the stage of training our network to fit the data.

Start with a network consisting of a single non-spiking output layer (i.e. spikes connected directly via a weight matrix to two non-spiking LIF neurons as output).

1. ⚠️ Find a good initialisation for this network that gives outputs roughly in the right range.
2. ⚠️ Write code to train your network to minimise the mean squared error between the output of your network and the recorded velocities. You will need to select an optimisation algorithm, learning rate, etc.
3. ⚠️ Train your network, plotting the loss curves for the training and validation data during training, and print out the test loss at the end.
4. ⚠️ Plot the output of your model and compare to the data for 8 randomly sampled time windows of length 1.

Don't worry too much about performance at the moment. You should be able to do better than the null loss, but you don't need to do hugely better. I get a null loss of around 1.04 and a test loss of around 0.55, but it's OK if you don't get such good results, and you may get much better results because I didn't do any hyperparameter searching.

You may want to use the following code as a starting point (it worked well enough for me but you can probably do better). On my desktop with CPU only, this took about two minutes to train.

In [18]:
# # Training parameters
# lr = 0.001 # learning rate
# num_epochs = 10
# max_num_batches = 40
# length = 1
# batch_size = 32 # small batches worked better for me for some reason

# # Optimiser and loss function
# optimizer = torch.optim.Adam(..., lr=lr) # what should the first argument be?

# # Training
# loss_hist = []
# val_loss_hist = []
# with tqdm(total=num_epochs*max_num_batches) as pbar:
#   last_epoch_loss = val_loss = null_val_loss = None
#   for epoch in range(num_epochs):
#     local_loss = []
#     for x, y in batched_data(...):
#       # Run the network
#       y_out = net(x)
#       # Compute a loss
#       loss = mse(y_out, y)
#       local_loss.append(loss.item())
#       # Update gradients
#       optimizer.zero_grad()
#       loss.backward()
#       optimizer.step()
#       # do something to ensure that taus remain positive, maybe set 1 ms as minimum value
#       pbar.update(1)
#       pbar.set_postfix(epoch=epoch, last_epoch_loss=last_epoch_loss, loss=loss.item(), val_loss=val_loss, null_val_loss=null_val_loss)
#     last_epoch_loss = np.mean(local_loss)
#     val_loss, null_val_loss = evaluate_network(net, ...)
#     pbar.set_postfix(epoch=epoch, last_epoch_loss=last_epoch_loss, loss=loss.item(), val_loss=val_loss, null_val_loss=null_val_loss)
#     loss_hist.append(last_epoch_loss)
#     val_loss_hist.append(val_loss)

# # Plot the loss function over time
# plt.semilogy(loss_hist, label='Testing loss')
# plt.semilogy(val_loss_hist, label='Validation loss')
# plt.axhline(null_val_loss, ls='--', c='r', label='Null model loss')
# plt.xlabel('Epoch')
# plt.ylabel('MSE')
# plt.legend(loc='best')
# plt.tight_layout()

# testing_loss, null_testing_loss = evaluate_network(net, testing_range, length=length, batch_size=batch_size)
# print(f'{testing_loss=}, {null_testing_loss=}')

# # Plot trained output
# plt.figure(figsize=(16, 6))
# with torch.no_grad():
#   for x, y in batched_data(..., batch_size=8, max_num_batches=1):
#     for b in range(8):
#       plt.subplot(2, 4, b+1)
#       y_out = net(x)
#       plt.plot(y_out[b, 0, :], ':C0', label='x_out')
#       plt.plot(y_out[b, 1, :], ':C1', label='y_out')
#       plt.plot(y[b, 0, :], '--C0', label='x')
#       plt.plot(y[b, 1, :], '--C1', label='y')
#       # Plot a smoothed version as well
#       plt.plot(savgol_filter(y_out[b, 0, :], 151, 3), '-C0', label='x_out (smooth)')
#       plt.plot(savgol_filter(y_out[b, 1, :], 151, 3), '-C1', label='y_out (smooth)')
#       plt.ylim(-5, 5)
# plt.tight_layout();

In [19]:
# Task 5: train a direct 52-input -> 2-non-spiking-output decoder.
# Independent checkpoints let the tests run without rerunning training.
import copy, time
models={}
models['single_layer.pt']=SNNNetwork([num_neurons,2],dt=training_dt,weight_scale=.025)
models['single_layer.pt'].fname='single_layer.pt'

def train_network(net,epochs=10,batches_per_epoch=40,lr=.003,
                  batch_size=32,dt=.01,clip=1.,verbose=True):
    optimiser=torch.optim.Adam(net.parameters(),lr=lr)
    train_hist,val_hist=[],[]
    tic=time.monotonic()
    for epoch in range(epochs):
        net.train();losses=[]
        for x,y in batched_data(training_range,dt=dt,length=1.,batch_size=batch_size,
                                max_num_batches=batches_per_epoch,seed=1000+epoch):
            optimiser.zero_grad(set_to_none=True)
            pred=net(x)
            loss=mse(pred,y)
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(),clip)
            optimiser.step()
            # Positive time constants enforced by log-parametrisation.
            with torch.no_grad():
                for layer in net.layers:
                    layer.log_tau.clamp_(np.log(.001),np.log(5.))
            losses.append(loss.item())
        if not losses: raise RuntimeError('No training batches generated.')
        tr=float(np.mean(losses))
        va,null=evaluate_network(net,validation_range,dt=dt,max_num_batches=15)
        train_hist.append(tr);val_hist.append(va)
        if verbose: print(f'Epoch {epoch+1}/{epochs}: train={tr:.4f}; val={va:.4f}; null={null:.4f}')
    if verbose:print('Elapsed seconds:',round(time.monotonic()-tic,1))
    return train_hist,val_hist

single_train,single_val=train_network(models['single_layer.pt'],epochs=10,
                                    batches_per_epoch=40,batch_size=32,lr=.003)
single_test,single_null=evaluate_network(models['single_layer.pt'],testing_range,
                                        dt=training_dt,max_num_batches=None)
print(f'HELD-OUT TEST MSE = {single_test:.4f}; null MSE = {single_null:.4f}')
assert parameters_are_finite(models['single_layer.pt'])
torch.save(models['single_layer.pt'].state_dict(),'single_layer_weights.pt')
plt.figure(figsize=(6,3));plt.plot(single_train,label='Train');plt.plot(single_val,label='Validation')
plt.axhline(single_null,color='k',ls='--',label='Null baseline')
plt.xlabel('Epoch');plt.ylabel('MSE');plt.title('Single-layer learning curve')
plt.legend();plt.tight_layout();plt.show()

@torch.no_grad()
def plot_sample_fits(net,blocks,dt=.01,seed=80):
    net.eval()
    x,y=next(batched_data(blocks,dt=dt,length=1.,batch_size=8,
                          max_num_batches=1,seed=seed))
    pred=net(x).numpy();actual=y.numpy()
    fig,axes=plt.subplots(2,4,figsize=(16,6),sharex=True)
    for b,ax in enumerate(axes.flat):
        for c,color in enumerate(('C0','C1')):
            ax.plot(np.arange(y.shape[-1])*dt,actual[b,c],color=color,ls='--',label=('x' if c==0 else 'y')+' actual')
            ax.plot(np.arange(y.shape[-1])*dt,pred[b,c],color=color,label=('x' if c==0 else 'y')+' predicted')
        if b==0:ax.legend(fontsize=7)
        ax.set_title(f'Window {b+1}')
    plt.tight_layout();plt.show()
plot_sample_fits(models['single_layer.pt'],testing_range,dt=training_dt)

Epoch 1/10: train=0.8025; val=0.6015; null=0.9723
Epoch 2/10: train=0.5855; val=0.5504; null=0.9723
Epoch 3/10: train=0.5380; val=0.5322; null=0.9723
Epoch 4/10: train=0.5584; val=0.5274; null=0.9723
Epoch 5/10: train=0.5278; val=0.5228; null=0.9723
Epoch 6/10: train=0.5118; val=0.5096; null=0.9723
Epoch 7/10: train=0.5048; val=0.5021; null=0.9723
Epoch 8/10: train=0.5108; val=0.4947; null=0.9723
Epoch 9/10: train=0.4821; val=0.4791; null=0.9723
Epoch 10/10: train=0.4805; val=0.4893; null=0.9723
Elapsed seconds: 57.8
HELD-OUT TEST MSE = 0.5358; null MSE = 1.0979


## Task 6: Longer length decoding

Your code above probably doesn't look great when plotted. That's partly because the outputs start at 0 but the data doesn't necessarily have to, so it takes a while for them to get in sync, and partly because on some intervals it will just do badly. To fix this, and to extend the fit to a longer range of data, in this task we only use the final timestep of each segment and compare to the data. Take a 15 second segment of testing data, and sample every 0.2 seconds to get 75 data points. For each data point, take a 1 second segment of time before this data point (these will be overlapping), run your simulation for that one second, and use the final time point of the simulated output as your prediction. Plot this compared to the real data for 8 different segments of 15 seconds.

This should look like a reasonable fit to the data. Congratulations, you have used raw spiking output of neurons recorded from a monkey's brain to predict what it was doing on a computer screen it was interacting with. That's a brain machine interface right here.

1. ⚠️ Implement the decoding plot and run it on your trained network.

You can use the template below to get you started.

Your results might look something like this:

![Fits](fits.png)

In [20]:
# def decoding_plot(net, dt_decoding=0.2, decoding_start=1000, decoding_length=15, length=1, dt=1e-3, figdims=(4, 2)):
#     w_intervals = np.arange(decoding_start+length, decoding_start+length+decoding_length, dt_decoding)
#     batch_size_single = len(w_intervals)-1
#     num_time_points = int(np.round(length/dt))
#     nfx, nfy = figdims
#     nf = nfx*nfy
#     batch_size = nf*batch_size_single
#     nrows = nfy*2
#     ncols = nfx
#     with torch.no_grad():
#         x = torch.zeros((batch_size, num_neurons, num_time_points))
#         y = torch.zeros((batch_size, 2, num_time_points))
#         T = []
#         for b in range(batch_size):
#            w_start = decoding_start+dt_decoding*b
#            w_end = w_start+length
#            T.append(w_end)
#            # ... (copy data to x, y)
#         T = np.array(T)
#         y_out = ...
#         mean_mse = mse(y, y_out)
#         plt.figure(figsize=(ncols*3, nrows*2))
#         for nf_i in range(nf):
#             sp_x = nf_i % nfx
#             sp_y = nf_i // nfx
#             for i in range(2):
#                 plt.subplot(nrows, ncols, sp_x+1+2*ncols*sp_y+i*ncols)
#                 # ...
#                 plt.ylim(-4, 4)
#                 if sp_x==0:
#                     plt.ylabel('Velocity')
#                 if 2*sp_y+i==nrows-1:
#                     plt.xlabel('Time (s)')
#                 if nf_i==0:
#                     plt.legend(loc='best')
#         plt.suptitle(f'{net.fname}: {mean_mse=:.3f}')
#         plt.tight_layout()

# decoding_plot(models['single_layer.pt'])

In [21]:
# Task 6: true rolling-window decoder, using ONLY last predicted timestep.
@torch.no_grad()
def decoding_plot(net,dt_decoding=.2,decoding_start=None,
                  decoding_length=15.,length=1.,dt=.01,figdims=(4,2)):
    net.eval()
    nfx,nfy=figdims
    fig,axes=plt.subplots(nfy*2,nfx,figsize=(16,3*nfy*2),sharey=True)
    if decoding_start is None:
        # Choose eight disjoint 16-second ranges, each entirely within a held-out block.
        if len(testing_range)<nfx*nfy:raise ValueError('Not enough test blocks')
        starts=testing_range[:nfx*nfy] + 1.
    else:
        starts=decoding_start+np.arange(nfx*nfy)*(decoding_length+length)
    err_sq=[]
    for k,start in enumerate(starts):
        # All windows end inside a single held-out block when starts are default.
        endpoints=start+length+np.arange(int(round(decoding_length/dt_decoding)))*dt_decoding
        batches=[]
        for end in endpoints:
            x,_=build_window(end-length,length,dt)
            batches.append(x)
        xb=torch.from_numpy(np.stack(batches))
        # Avoid storing a giant computational graph or full outputs for every window.
        pred=np.concatenate([net(chunk)[:,:,-1].numpy() for chunk in xb.split(32)],axis=0)
        true=np.column_stack([np.interp(endpoints,vel_times,vel[:,c]) for c in range(2)])
        err_sq.append((pred-true)**2)
        for c in range(2):
            ax=axes[2*(k//nfx)+c,k%nfx]
            ax.plot(endpoints,true[:,c],'--',color=f'C{c}',label=('x' if c==0 else 'y')+' actual')
            ax.plot(endpoints,pred[:,c],color=f'C{c}',label=('x' if c==0 else 'y')+' estimated')
            ax.set_ylim(-4,4)
            if k%nfx==0:ax.set_ylabel('Velocity')
            if 2*(k//nfx)+c==nfy*2-1:ax.set_xlabel('Time (s)')
            if k==0:ax.legend(fontsize=8)
    rolling_mse=float(np.mean(np.concatenate(err_sq)))
    fig.suptitle(f'{net.fname}: 15-second rolling decoding MSE={rolling_mse:.3f}')
    plt.tight_layout();plt.show()
    print(f'Rolling-window MSE (last time step only): {rolling_mse:.4f}')
    return rolling_mse

single_rolling_mse=decoding_plot(models['single_layer.pt'],dt=training_dt)

Rolling-window MSE (last time step only): 0.5971


## Task 7: Comparing spiking and non-spiking

1. ⚠️ Now try training your network with at least one spiking hidden layer. Compare your results to the non-spiking version. Note that training times with a spiking hidden layer are likely to be much longer. My training time went up from 2 minutes to 30 minutes (with CPU only).
2. ⚠️ Plot the trained distribution of weights and time constants for both models.

You might find it interesting to compare the distribution of time constants you find for the spiking hidden layer with the results in our paper [Perez et al. 2021](https://neural-reckoning.org/pub_heterogeneity.html). Note: this is not required for credit, only for your interest.

Well done - you have made it to the end!

In [22]:
# Task 7: spiking hidden layer, identical input/output and split protocol.
models['spiking_hidden.pt']=SNNNetwork([num_neurons,100,2],dt=training_dt,
                                      hidden_spiking=True,weight_scale=.4)
models['spiking_hidden.pt'].fname='spiking_hidden.pt'
# The hidden layer needs larger impulses to fire; the readout needs much smaller
# weights to prevent summing 100 spike trains into an enormous output.
with torch.no_grad():
    models['spiking_hidden.pt'].layers[-1].w.mul_(.025)
spiking_train,spiking_val=train_network(models['spiking_hidden.pt'],epochs=6,
                                       batches_per_epoch=8,batch_size=8,lr=.002)
spiking_test,spiking_null=evaluate_network(models['spiking_hidden.pt'],testing_range,
                                          dt=training_dt,max_num_batches=None)
print(f'Spiking model held-out MSE={spiking_test:.4f}, null={spiking_null:.4f}')
assert parameters_are_finite(models['spiking_hidden.pt'])
torch.save(models['spiking_hidden.pt'].state_dict(),'spiking_hidden_weights.pt')
plt.figure(figsize=(6,3))
plt.plot(single_val,label='Direct / non-spiking output')
plt.plot(np.linspace(0,len(single_val)-1,len(spiking_val)),spiking_val,label='Spiking hidden')
plt.axhline(single_null,ls='--',color='k',label='Null')
plt.xlabel('Relative training progression');plt.ylabel('Validation MSE')
plt.legend();plt.tight_layout();plt.show()

fig,axes=plt.subplots(2,2,figsize=(12,6))
for name,model in models.items():
    weights=np.concatenate([l.w.detach().numpy().ravel() for l in model.layers])
    taus=np.concatenate([l.tau.detach().numpy().ravel() for l in model.layers])*1e3
    axes[0,0].hist(weights,bins=40,alpha=.55,label=name)
    axes[0,1].hist(taus,bins=35,alpha=.55,label=name)
    for i,l in enumerate(model.layers):
        axes[1,0].hist(l.w.detach().numpy().ravel(),bins=35,alpha=.35,label=f'{name}: layer {i+1}')
        axes[1,1].hist((l.tau.detach().numpy()*1e3),bins=30,alpha=.35,label=f'{name}: layer {i+1}')
axes[0,0].set(xlabel='Weight',ylabel='Count',title='Trained weight distribution')
axes[0,1].set(xlabel='Time constant (ms)',ylabel='Count',title='Trained tau distribution')
axes[1,0].set(xlabel='Weight',ylabel='Count',title='Per-layer weight distributions')
axes[1,1].set(xlabel='Time constant (ms)',ylabel='Count',title='Per-layer tau distributions')
for ax in axes.flat:ax.legend(fontsize=6)
plt.tight_layout();plt.show()
print('Held-out test comparison:')
print(f'  Direct decoder:  MSE={single_test:.4f} (null={single_null:.4f})')
print(f'  Spiking-hidden: MSE={spiking_test:.4f} (null={spiking_null:.4f})')
plot_network_activity(models['spiking_hidden.pt'],testing_range,dt=training_dt)
plot_sample_fits(models['spiking_hidden.pt'],testing_range,dt=training_dt)

Epoch 1/6: train=1.2612; val=0.8737; null=0.9723
Epoch 2/6: train=0.9063; val=1.0618; null=0.9723
Epoch 3/6: train=0.8711; val=0.7711; null=0.9723
Epoch 4/6: train=0.8693; val=0.6849; null=0.9723
Epoch 5/6: train=0.6738; val=0.7887; null=0.9723
Epoch 6/6: train=0.8314; val=0.7528; null=0.9723
Elapsed seconds: 9.6
Spiking model held-out MSE=0.8251, null=1.0979
Held-out test comparison:
  Direct decoder:  MSE=0.5358 (null=1.0979)
  Spiking-hidden: MSE=0.8251 (null=1.0979)
Hidden firing rates (Hz): mean 13.09 min 0.0 max 71.0


**Comparison protocol.** Both models use the same underlying block-separated train/validation/test splits, the same simulation step and the same MSE evaluation. Their network sizes, training budgets and optimisation trajectories differ, so the comparison is illustrative rather than a controlled claim of architectural superiority. Compare held-out loss against the null baseline and the weight/tau distributions. The feedback marks require substantive peer feedback delivered separately; this notebook cannot earn those marks by itself.